# Apache Spark II, Making Spark Fast: Shuffles, Skew, Broadcast, Caching, AQE and the Spark UI

Notebook for **Chapter 10** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch10.html

Tip: change a number, predict the output, then run the cell again.

## 10.1 Shuffles and Partition Sizing

## 10.2 Skew, Salting, Broadcast Joins and AQE

In [ ]:
import random, zlib, collections, statistics
random.seed(1)
P=8; N=100000
keys=["hot"]*(N//2)+[f"user{random.randint(1,5000)}" for _ in range(N//2)]
def part(k): return zlib.crc32(k.encode())%P     # deterministic hash
def sizes(ks):
    c=collections.Counter(part(k) for k in ks); return [c.get(i,0) for i in range(P)]
s=sizes(keys); print("plain hash partition sizes:",s)
print(f"max={max(s)} mean={statistics.mean(s):.0f} max/mean={max(s)/statistics.mean(s):.2f}")
S=8
salted=[f"{k}#{random.randrange(S)}" if k=="hot" else k for k in keys]
s2=sizes(salted); print("salted sizes             :",s2)
print(f"max={max(s2)} mean={statistics.mean(s2):.0f} max/mean={max(s2)/statistics.mean(s2):.2f}")
for lab,ss in (("plain",s),("salted",s2)):
    print(f"{lab}: stage time = max partition / 10000 rows/s = {max(ss)/10000:.2f} s  (ideal {N/P/10000:.2f} s)")

In [ ]:
# not run here
salted = df.withColumn("salt", (F.rand() * 8).cast("int"))
step1 = salted.groupBy("user_id", "salt").agg(F.sum("amount").alias("part"))
final = step1.groupBy("user_id").agg(F.sum("part").alias("total"))

In [ ]:
# not run here
from pyspark.sql.functions import broadcast
out = events.join(broadcast(devices), "device_id")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 100 * 1024 * 1024)  # 100 MB

## 10.3 Caching, Memory and Reading the Spark UI

## 10.4 AQE Walked Through and a Spark UI Reading Drill

Section 10.2 listed what Adaptive Query Execution does. Here you will do what AQE does, by hand and in code, on one set of numbers, and then practise the other half of the skill: looking at a stage's task numbers and saying what is wrong. Real AQE is more careful than our toy (it also looks at map-side statistics and respects ordering rules), but the three decisions are the same.

In [ ]:
# AQE by hand: the three decisions, using real partition sizes (MB) from a finished shuffle stage.
import statistics
advisory = 64
sizes = [3, 2, 5, 1, 4, 2, 6, 3, 2, 1, 4, 5, 3, 2, 800, 4]   # 16 shuffle partitions, one is huge

# 1) coalesce: walk the partitions in order, start a new group when adding would pass the advisory size
groups, cur, cur_sum = [], [], 0
for i, s in enumerate(sizes):
    if cur and cur_sum + s > advisory:
        groups.append(cur); cur, cur_sum = [], 0
    cur.append(i); cur_sum += s
groups.append(cur)
print("partitions before:", len(sizes), " tasks after coalescing:", len(groups))
for g in groups:
    print("  group", g[0], "to", g[-1], "=", sum(sizes[i] for i in g), "MB")

# 2) skew split: a partition is skewed if > factor * median AND > threshold
factor, threshold = 5, 256
med = statistics.median(sizes)
skewed = [i for i, s in enumerate(sizes) if s > factor * med and s > threshold]
print("median", med, "MB; skewed partitions:", skewed)
for i in skewed:
    pieces = -(-sizes[i] // advisory)       # ceiling division: split into advisory sized pieces
    print(f"  partition {i}: {sizes[i]} MB -> {pieces} sub-tasks of about {sizes[i]/pieces:.0f} MB")
    print("  (the other join side's matching partition is read once per piece)")

# 3) join switching: after a filter the real size of one side is tiny, so use broadcast
broadcast_threshold = 10   # MB
planned_right_size = 4000  # MB, the estimate used at planning time (no statistics)
actual_right_after_filter = 7
print("planned join:", "sort-merge" if planned_right_size > broadcast_threshold else "broadcast")
print("after stage finished, actual size", actual_right_after_filter, "MB ->",
      "broadcast" if actual_right_after_filter <= broadcast_threshold else "sort-merge")

partitions before: 16  tasks after coalescing: 3
  group 0 to 13 = 43 MB
  group 14 to 14 = 800 MB
  group 15 to 15 = 4 MB
median 3.0 MB; skewed partitions: [14]
  partition 14: 800 MB -> 13 sub-tasks of about 62 MB
  (the other join side's matching partition is read once per piece)
planned join: sort-merge
after stage finished, actual size 7 MB -> broadcast


In [ ]:
# Reading a Spark UI stage summary: tasks' durations (seconds) from three runs of the same stage.
import statistics
runs = {
  "A skewed":      [4, 5, 4, 5, 6, 5, 4, 5, 5, 4, 5, 190],
  "B healthy":     [5, 6, 5, 5, 6, 7, 5, 6, 5, 6, 5, 6],
  "C too many":    [0.2] * 400,
}
for name, d in runs.items():
    q = statistics.quantiles(d, n=4)
    mx, med = max(d), statistics.median(d)
    total = sum(d)
    print(f"{name:11s} tasks={len(d):3d} median={med:5.2f} max={mx:6.1f} max/median={mx/med:5.1f} "
          f"sum={total:6.1f}s")
# wall time with 12 cores: tasks run in waves; the stage ends when the last finishes
def wall(d, cores):
    free = [0.0] * cores
    for t in d:
        i = free.index(min(free)); free[i] += t
    return max(free)
for name, d in runs.items():
    print(name, "wall time on 12 cores:", round(wall(d, 12), 1), "s; ideal:", round(sum(d)/12, 1), "s")

A skewed    tasks= 12 median= 5.00 max= 190.0 max/median= 38.0 sum= 242.0s
B healthy   tasks= 12 median= 5.50 max=   7.0 max/median=  1.3 sum=  67.0s
C too many  tasks=400 median= 0.20 max=   0.2 max/median=  1.0 sum=  80.0s
A skewed wall time on 12 cores: 190.0 s; ideal: 20.2 s
B healthy wall time on 12 cores: 7.0 s; ideal: 5.6 s
C too many wall time on 12 cores: 6.8 s; ideal: 6.7 s


## Chapter recap

## Practice set

**1.** (easy) What does `Exchange` mean in a physical plan? **Answer:** a shuffle, and the boundary between two stages.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch10.html